In [1]:
from julia.api import Julia
from julia import Main

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures

import scipy.special as sp
import os

import pathlib
from pathlib import Path
import json

#fitters

import pybobyqa
import time
import cma
import csv

def to_float64(df):
    num_cols = df.select_dtypes(include=["number"]).columns
    df[num_cols] = df[num_cols].astype("float64")
    return df

Which Fit?

In [2]:
fit_name = "Default"

approximate_total_xsec = True
data_uncertainty_only = False  # use experimental + PDF covariance in chi2

N_replicas = 60
vary_data_points = True
use_pdf_shift = True
pdf_shift_mode = "sequential"  # "random" or "sequential"

output_csv_name = "replica_0311.csv"
append_to_existing_csv = True
use_random_seed = True
replica_seed = 12345


Read Files

In [3]:
TMD_fitting_root = "../"
def include(name):
    path = os.path.join(TMD_fitting_root, name)
    Main.eval(f'include(raw"{path}")')

include(f"Cards/{fit_name}.jl")
include(f"DY/DY_table_{Main.flavor_scheme}.jl")

# Data
file_root = f"../Data/{Main.data_name}/Cutted/DY"
matrix_root = f"../Data/{Main.data_name}/Covariance_Matrices/DY"
table_root = f"../Tables/{Main.table_name}/DY"
total_root = f"../Data/DY_total_xsec/{Main.pdf_name}"
error_sets_root = f"../Data/PDF_Matrices/{Main.error_sets_name}/DY"
pdf_diff_root = f"../Data/PDF_Differences/{Main.error_sets_name}"

initial_params = Main.initial_params


By file or by experiment?

In [4]:
data_selections = "by_experiment"  # "by_file" or "by_experiment"

In [5]:
experiments =[
    'ATLAS_7',
    'ATLAS_8', 
    #'ATLAS_13', 
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'E605',
    'E772',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    #'PHENIX',
    'STAR'
]



#["E288","E605","E772","ATLAS", "CMS", "D0", "CDF", "LHCb", "PHENIX", "STAR"]

if data_selections == "by_file":
    file_names = [

    #----------------------------------------------------------------------------
    # ATLAS
    #----------------------------------------------------------------------------

    #"ATLAS/ATLAS_7TeV_y_0_1.csv",
    #"ATLAS/ATLAS_7TeV_y_1_2.csv",
    #"ATLAS/ATLAS_7TeV_y_2_2.4.csv",

    #"ATLAS/ATLAS_8TeV_Q_44_66.csv",
    #"ATLAS/ATLAS_8TeV_Q_116_150.csv",

    #"ATLAS/ATLAS_8TeV_y_0_0.4.csv"
    #"ATLAS/ATLAS_8TeV_y_0.4_0.8.csv"
    #"ATLAS/ATLAS_8TeV_y_0.8_1.2.csv"
    #"ATLAS/ATLAS_8TeV_y_1.2_1.6.csv",
    #"ATLAS/ATLAS_8TeV_y_1.6_2.csv",
    #"ATLAS/ATLAS_8TeV_y_2_2.4.csv",

    #----------------------------------------------------------------------------
    # CDF
    #----------------------------------------------------------------------------

    #"CDF/CDF_RunI.csv",
    #"CDF/CDF_RunII.csv",

    #----------------------------------------------------------------------------
    # CMS
    #----------------------------------------------------------------------------

    #"CMS/CMS_7TeV.csv",
    #"CMS/CMS_8TeV.csv",
    
    #"CMS/CMS_13TeV_y_0_0.4.csv",
    #"CMS/CMS_13TeV_y_0.4_0.8.csv",
    #"CMS/CMS_13TeV_y_0.8_1.2.csv",
    #"CMS/CMS_13TeV_y_1.2_1.6.csv",
    #"CMS/CMS_13TeV_y_1.6_2.4.csv",

    #----------------------------------------------------------------------------
    # D0
    #----------------------------------------------------------------------------

    #"D0/D0_RunI.csv",
    #"D0/D0_RunII.csv",
    #"D0/D0_RunIImu.csv",

    #----------------------------------------------------------------------------
    # LHCb
    #----------------------------------------------------------------------------

    #"LHCb/LHCb_7TeV.csv",
    #"LHCb/LHCb_8TeV.csv",
    #"LHCb/LHCb_13TeV.csv",

    #----------------------------------------------------------------------------
    # Phenix
    #----------------------------------------------------------------------------

    #"PHENIX/PHENIX_200.csv",

    #----------------------------------------------------------------------------
    # STAR
    #----------------------------------------------------------------------------

    #"STAR/STAR_510.csv",

    #----------------------------------------------------------------------------
    # E288
    #----------------------------------------------------------------------------

    #"E288/E288_200_Q_4_5.csv",
    #"E288/E288_200_Q_5_6.csv",
    #"E288/E288_200_Q_6_7.csv",
    #"E288/E288_200_Q_7_8.csv",
    #"E288/E288_200_Q_8_9.csv",
    #"E288/E288_200_Q_10_11.csv",

    #"E288/E288_300_Q_4_5.csv",
    #"E288/E288_300_Q_5_6.csv",
    #"E288/E288_300_Q_6_7.csv",
    #"E288/E288_300_Q_7_8.csv",
    #"E288/E288_300_Q_8_9.csv",
    #"E288/E288_300_Q_10_11.csv",
    #"E288/E288_300_Q_11_12.csv",

    #"E288/E288_400_Q_5_6.csv",
    #"E288/E288_400_Q_6_7.csv",
    #"E288/E288_400_Q_7_8.csv",
    #"E288/E288_400_Q_8_9.csv",
    #"E288/E288_400_Q_10_11.csv",
    #"E288/E288_400_Q_11_12.csv",
    #"E288/E288_400_Q_12_13.csv",
    #"E288/E288_400_Q_13_14.csv",

    #----------------------------------------------------------------------------
    # E605
    #----------------------------------------------------------------------------

    #"E605/E605_Q_7_8.csv",
    #"E605/E605_Q_8_9.csv",
    #"E605/E605_Q_10.5_11.5.csv",
    #"E605/E605_Q_11.5_13.5.csv",
    #"E605/E605_Q_13.5_18.csv",

    #----------------------------------------------------------------------------
    # E772
    #----------------------------------------------------------------------------

    #"E772/E772_Q_5_6.csv",
    #"E772/E772_Q_6_7.csv",
    #"E772/E772_Q_7_8.csv",
    #"E772/E772_Q_8_9.csv",
    #"E772/E772_Q_11_12.csv",
    #"E772/E772_Q_12_13.csv",
    #"E772/E772_Q_13_14.csv",
    #"E772/E772_Q_14_15.csv",
    ]

In [6]:
from pathlib import Path

file_excludes = [
    "E772/E772-5Q6.csv",
    "E772/E772-6Q7.csv",
    "E772/E772-7Q8.csv",
    "E772/E772-8Q9.csv",
]

if data_selections == "by_experiment":
    file_names = []
    for experiment in experiments:
        exp_dir = Path(file_root) / experiment
        for p in sorted(exp_dir.glob("*.csv")):
            if str(experiment + "/" + p.name) in file_excludes:
                continue
            file_names.append(str(Path(experiment) / p.name)) 

display(file_names)

['ATLAS_7\\ATLAS7-00y10.csv',
 'ATLAS_7\\ATLAS7-10y20.csv',
 'ATLAS_7\\ATLAS7-20y24.csv',
 'ATLAS_8\\ATLAS8-00y04.csv',
 'ATLAS_8\\ATLAS8-04y08.csv',
 'ATLAS_8\\ATLAS8-08y12.csv',
 'ATLAS_8\\ATLAS8-116Q150.csv',
 'ATLAS_8\\ATLAS8-12y16.csv',
 'ATLAS_8\\ATLAS8-16y20.csv',
 'ATLAS_8\\ATLAS8-20y24.csv',
 'ATLAS_8\\ATLAS8-46Q66.csv',
 'CDF_I\\CDF1.csv',
 'CDF_II\\CDF2.csv',
 'CMS_7\\CMS7.csv',
 'CMS_8\\CMS8.csv',
 'CMS_13\\CMS13-00y04.csv',
 'CMS_13\\CMS13-04y08.csv',
 'CMS_13\\CMS13-08y12.csv',
 'CMS_13\\CMS13-106Q170.csv',
 'CMS_13\\CMS13-12y16.csv',
 'CMS_13\\CMS13-16y24.csv',
 'CMS_13\\CMS13-170Q350.csv',
 'CMS_13\\CMS13-350Q1000.csv',
 'D0_I\\D01.csv',
 'D0_II\\D02.csv',
 'D0_II_mu\\D02mu.csv',
 'E288\\E228-200-4Q5.csv',
 'E288\\E228-200-5Q6.csv',
 'E288\\E228-200-6Q7.csv',
 'E288\\E228-200-7Q8.csv',
 'E288\\E228-200-8Q9.csv',
 'E288\\E228-300-11Q12.csv',
 'E288\\E228-300-4Q5.csv',
 'E288\\E228-300-5Q6.csv',
 'E288\\E228-300-6Q7.csv',
 'E288\\E228-300-7Q8.csv',
 'E288\\E228-300-8Q9.cs

Read Data

In [7]:
data_list = dict()
matrix_data_list = dict()
matrix_total_list = dict()
df_total_xsec = to_float64(pd.read_csv(f"{total_root}.csv"))
total_xsec_names = df_total_xsec['name'].tolist()

for file in tqdm(file_names):

    df_data = to_float64(pd.read_csv(f"{file_root}/{file}"))
    data_list[file] = df_data
    
    matrix_data = to_float64(pd.read_csv(f"{matrix_root}/{file}"))
    
    if data_uncertainty_only == True:
        matrix_total = matrix_data
    else:
        matrix_PDF = to_float64(pd.read_csv(f"{error_sets_root}/{file}"))
        matrix_total = matrix_data + matrix_PDF
    
    matrix_data_list[file] = matrix_data
    matrix_total_list[file] = matrix_total

    name_short = Path(file).stem
    if name_short in total_xsec_names:
        total_xsec = df_total_xsec[df_total_xsec['name'] == name_short]["total_xsec"].values[0]
        data_list[file]['total_xsec'] = total_xsec*np.ones(len(data_list[file]))
        print(f"{name_short}'s total xsec = {total_xsec} added")

 42%|████▏     | 24/57 [00:00<00:00, 183.94it/s]

ATLAS7-00y10's total xsec = 1.0 added
ATLAS7-10y20's total xsec = 1.0 added
ATLAS7-20y24's total xsec = 1.0 added
CMS7's total xsec = 1.0 added
CMS8's total xsec = 1.0 added
D02's total xsec = 1.0 added
D02mu's total xsec = 1.0 added


100%|██████████| 57/57 [00:00<00:00, 216.83it/s]


Prediction

In [8]:
Params = Main.Params_Struct(*[np.float32(x) for x in initial_params]) 
#Main.set_params(Main.VRAM, Params) 

for i in range(10):
    Params = Main.Params_Struct(*[np.float32(x) for x in initial_params])                  
    predictions,t = Main.xsec_dict(Main.rel_paths, Main.VRAM)
    print(round(t*1000,2), "ms")

279.18 ms
39.69 ms
38.96 ms
38.84 ms
39.03 ms
39.85 ms
39.08 ms
38.97 ms
39.0 ms
38.97 ms


In [9]:
def get_file_length():

    file_lengths = dict()

    for file in file_names:

        df = to_float64(pd.read_csv(f"{file_root}/{file}"))

        file_lengths[file] = len(df)

    return file_lengths

file_lengths = get_file_length()

In [10]:
def _norm(p: str) -> str:
    return os.path.normpath(p).replace('\\', '/')

df_total_xsec = to_float64(pd.read_csv(f"{total_root}.csv"))
total_xsec_names = df_total_xsec['name'].tolist()

def prediction_reformat(predictions):
    preds = {_norm(k): v for k, v in predictions.items()}  # normalize keys once
    df_predictions = {}

    for file in file_names:
        n = file_lengths[file]
        base = os.path.splitext(file)[0]
        xs = []
        for i in range(n):
            table_path = _norm(os.path.join(table_root, f"{base}/{i}.jls"))
            xs.append(preds[table_path])
        df_predictions[file] = np.array(xs)

        if approximate_total_xsec == True and Path(file).stem in total_xsec_names:
            data_xsec = data_list[file]["xsec"].to_numpy()
            dσ = sum(data_xsec)/sum(xs)
            df_predictions[file] = np.array(dσ * np.array(xs))
            #display(file, 1/dσ)

    return df_predictions

df_predictions = prediction_reformat(predictions)

In [11]:
import pickle

def clone_data_list(source):
    return {file: df.copy(deep=True) for file, df in source.items()}

nominal_data_list = clone_data_list(data_list)

def _pdf_dataset_key(file):
    return str(Path(file).with_suffix("")).replace("\\", "/")

def load_pdf_shift_replica(path):
    with path.open("rb") as f:
        pdf_diff_dict = pickle.load(f)

    pdf_shift_list = {}
    for file in file_names:
        prefix = f"{_pdf_dataset_key(file)}/"
        values = []
        for i in range(file_lengths[file]):
            key = f"{prefix}{i}"
            if key not in pdf_diff_dict:
                raise KeyError(f"Missing PDF-difference entry: {key}")
            values.append(pdf_diff_dict[key])
        pdf_shift_list[file] = np.asarray(values, dtype=np.float64)

    return pdf_shift_list

zero_pdf_shift_list = {
    file: np.zeros(file_lengths[file], dtype=np.float64)
    for file in file_names
}

pdf_shift_replicas = {}
pdf_replica_ids = np.array([], dtype=int)
if use_pdf_shift:
    pdf_diff_paths = sorted(Path(pdf_diff_root).glob("*.pkl"), key=lambda p: int(p.stem))
    if not pdf_diff_paths:
        raise FileNotFoundError(f"No PDF-difference replicas found under {pdf_diff_root}")

    pdf_shift_replicas = {
        int(path.stem): load_pdf_shift_replica(path)
        for path in tqdm(pdf_diff_paths, desc="Loading PDF-difference replicas")
    }
    pdf_replica_ids = np.array(sorted(pdf_shift_replicas), dtype=int)
    print(f"Loaded {len(pdf_replica_ids)} PDF-difference replicas from {pdf_diff_root}")
else:
    print("PDF-difference replica shifts are disabled.")

current_pdf_replica_id = None
current_pdf_shift_list = zero_pdf_shift_list

def apply_pdf_shift(df_predictions, pdf_shift_list):
    shifted = {}
    for file in file_names:
        shifted[file] = np.asarray(df_predictions[file], dtype=np.float64) + pdf_shift_list[file]
    return shifted


Loading PDF-difference replicas: 100%|██████████| 100/100 [00:00<00:00, 2882.13it/s]

Loaded 100 PDF-difference replicas from ../Data/PDF_Differences/MSHT20N3LO-MC


Chi2

In [12]:
ASWZ_b_array = np.linspace(0.12,0.78,12)*5.067731
ASWZ_prediction = np.array([
    -0.08158508158508182,
    -0.1701631701631705,
    -0.2400932400932403,
    -0.34265734265734293,
    -0.37062937062937085,
    -0.4265734265734267,
    -0.4498834498834501,
    -0.44522144522144536,
    -0.4965034965034967,
    -0.5710955710955714,
    -0.6363636363636365,
    -0.7016317016317017
    ])
ASWZ_upper = np.array([
    0.18414918414918402,
    0.11421911421911402,
    0.09557109557109533,
    0.002331002331002141,
    0.016317016317016098,
    -0.034965034965035224,
    -0.034965034965035224,
    -0.011655011655011815,
    -0.034965034965035224,
    -0.05361305361305391,
    -0.05827505827505863,
    -0.04895104895104918
    ])
ASWZ_error = np.array(ASWZ_upper) - np.array(ASWZ_prediction)

def chi2_lattice(): 
    CS_list = []
    for b in ASWZ_b_array :
        Q = 2.0
        CS = Main.CS_total_func(b, Q)
        CS_list.append(CS)
    chi2dN = np.sum( (CS_list - ASWZ_prediction)**2 / ASWZ_error**2 ) / len(ASWZ_b_array)
    return chi2dN

def timed(func):
    t0 = time.perf_counter()
    out = func()
    return out, time.perf_counter() - t0

#chi2dN, t = timed(chi2_lattice)
#print("χ^2/N from LATTICE =", chi2dN, ", took", round(t, 4), "seconds")

In [13]:
def get_chi2dN(df_predictions):

    N_list = dict()
    chi2dN_list = dict()
    chi2_total = 0.0
    N_total = 0

    for file in file_names:

        data_xsec = data_list[file]["xsec"].to_numpy()
        pred_xsec = df_predictions[file]
        diff_xsec = data_xsec - pred_xsec

        covariance_matrix_inv = np.linalg.inv(matrix_total_list[file])

        N = len(data_xsec)

        chi2 = diff_xsec @ covariance_matrix_inv @ diff_xsec

        chi2_total += chi2
        N_total += N
        chi2dN_list[file] = float(round(chi2/N, 3))
        N_list[file] = N

    chi2dN = chi2_total / N_total
    return chi2dN, chi2dN_list, N_list

chi2dN, chi2dN_list, N_list = get_chi2dN(df_predictions)

print(f"Total χ^2/N = {chi2dN:.3f}")
display(chi2dN_list)

Total χ^2/N = 0.829


{'ATLAS_7\\ATLAS7-00y10.csv': 1.21,
 'ATLAS_7\\ATLAS7-10y20.csv': 4.613,
 'ATLAS_7\\ATLAS7-20y24.csv': 1.767,
 'ATLAS_8\\ATLAS8-00y04.csv': 3.175,
 'ATLAS_8\\ATLAS8-04y08.csv': 1.509,
 'ATLAS_8\\ATLAS8-08y12.csv': 1.598,
 'ATLAS_8\\ATLAS8-116Q150.csv': 0.456,
 'ATLAS_8\\ATLAS8-12y16.csv': 1.141,
 'ATLAS_8\\ATLAS8-16y20.csv': 1.506,
 'ATLAS_8\\ATLAS8-20y24.csv': 1.013,
 'ATLAS_8\\ATLAS8-46Q66.csv': 0.43,
 'CDF_I\\CDF1.csv': 0.501,
 'CDF_II\\CDF2.csv': 0.811,
 'CMS_7\\CMS7.csv': 1.59,
 'CMS_8\\CMS8.csv': 0.886,
 'CMS_13\\CMS13-00y04.csv': 1.506,
 'CMS_13\\CMS13-04y08.csv': 0.913,
 'CMS_13\\CMS13-08y12.csv': 0.523,
 'CMS_13\\CMS13-106Q170.csv': 0.349,
 'CMS_13\\CMS13-12y16.csv': 0.321,
 'CMS_13\\CMS13-16y24.csv': 0.289,
 'CMS_13\\CMS13-170Q350.csv': 0.674,
 'CMS_13\\CMS13-350Q1000.csv': 1.254,
 'D0_I\\D01.csv': 0.555,
 'D0_II\\D02.csv': 0.267,
 'D0_II_mu\\D02mu.csv': 0.199,
 'E288\\E228-200-4Q5.csv': 0.341,
 'E288\\E228-200-5Q6.csv': 0.355,
 'E288\\E228-200-6Q7.csv': 0.441,
 'E288\\E228-2

Objective

In [14]:
def objective(params):
    try:
        params_cl = Main.Params_Struct(*[np.float32(x) for x in params])
        Main.set_params(Main.VRAM, params_cl)

        predictions, t = Main.xsec_dict(Main.rel_paths, Main.VRAM)

        df_predictions = prediction_reformat(predictions)
        df_predictions = apply_pdf_shift(df_predictions, current_pdf_shift_list)
        chi2dN, _, _ = get_chi2dN(df_predictions)

        if not np.isfinite(chi2dN):
            return 1e5
        return chi2dN

    except Exception as e:
        return 1e5

print(objective(initial_params))

# freeze parameters

initial_params = np.asarray(initial_params, float)
frozen_idx = np.asarray(Main.frozen_indices, int)

dim_full = len(initial_params)
mask = np.ones(dim_full, dtype=bool)
mask[frozen_idx] = False
free_idx = np.where(mask)[0]   # indices that WILL be fitted

frozen_vals = initial_params[frozen_idx].copy()

def objective_with_freeze(params_free):

    params_free = np.asarray(params_free, float)
    full = initial_params.copy()
    full[free_idx] = params_free
    full[frozen_idx] = frozen_vals
    return objective(full)

print(objective_with_freeze(initial_params[free_idx]))


0.829330721673905
0.8293308603154625


Bounds

In [15]:
bounds_raw = np.asarray(Main.bounds_raw, float)[free_idx]

lower_bounds, upper_bounds = np.array(list(zip(*bounds_raw)))

def objective_normalized(params):

    normalized_params = lower_bounds + params * (upper_bounds - lower_bounds)

    return objective_with_freeze(normalized_params)

def objective_log(params):
    return np.log10(objective_normalized(params))

def normalize_params(params):
    return (params - lower_bounds) / (upper_bounds - lower_bounds)

def denormalize_params(params):
    return lower_bounds + params * (upper_bounds - lower_bounds)

theta0 = normalize_params(np.array(initial_params)[free_idx])
dim = len(bounds_raw)

## Replica Refit Workflow

Each fit replica draws:

1. A pseudo-dataset from the experimental covariance matrix.
2. A random PDF-difference replica from `Data/PDF_Differences/{error_sets_name}`.

The chosen PDF-difference vector is added pointwise to the theory prediction before evaluating `χ²`.


In [16]:
if use_random_seed:
    replica_rng = np.random.default_rng()
else:
    replica_rng = np.random.default_rng(replica_seed)

randomize_start = False
alpha = 9.0

maxfun = 5000
rhobeg = 0.15
rhoend = 1e-5

results_path = Path("replica_data/",output_csv_name)

param_columns = [f"param_{i}" for i in range(dim_full)]


In [17]:
def generate_experimental_replica_data(rng):
    replica_data = clone_data_list(nominal_data_list)
    if not vary_data_points:
        return replica_data

    for file in file_names:
        xsecs = nominal_data_list[file]["xsec"].to_numpy(np.float64)
        cov = np.asarray(matrix_data_list[file], np.float64)
        cov = 0.5 * (cov + cov.T)

        eps = np.finfo(cov.dtype).eps * max(1.0, float(np.max(np.diag(cov))))
        L = np.linalg.cholesky(cov + eps * np.eye(len(xsecs)))

        replica_data[file]["xsec"] = xsecs + L @ rng.standard_normal(len(xsecs))

    return replica_data

def sample_pdf_replica(rng, replica_id):
    if not use_pdf_shift:
        return None, zero_pdf_shift_list

    if pdf_shift_mode == "random":
        pdf_replica_id = int(rng.choice(pdf_replica_ids))
    elif pdf_shift_mode == "sequential":
        pdf_replica_id = int(replica_id) + 1
        if pdf_replica_id not in pdf_shift_replicas:
            raise KeyError(
                f"Sequential PDF-shift mode requires PDF replica {pdf_replica_id} for fit replica {replica_id}, "
                f"but it was not found under {pdf_diff_root}"
            )
    else:
        raise ValueError(
            f"Unknown pdf_shift_mode={pdf_shift_mode!r}; use 'random' or 'sequential'"
        )

    return pdf_replica_id, pdf_shift_replicas[pdf_replica_id]

def draw_theta0(rng):
    if not randomize_start:
        return theta0.copy()

    while True:
        cand = rng.beta(alpha, alpha, size=dim)
        val = float(objective_log(cand))
        if val < 1.0:
            return cand

def prepare_results_file(path):
    header = ["replica_id", "pdf_replica_id", "success", "nfev", "best_chi2dN", *param_columns]
    with path.open("w", newline="") as f:
        csv.writer(f).writerow(header)

def get_start_replica_id(path):
    if (not append_to_existing_csv) or (not path.exists()) or path.stat().st_size == 0:
        return 0

    with path.open("r", newline="") as f:
        rows = list(csv.DictReader(f))

    if not rows:
        return 0

    return max(int(row["replica_id"]) for row in rows) + 1

if (not append_to_existing_csv) or (not results_path.exists()) or results_path.stat().st_size == 0:
    prepare_results_file(results_path)

start_replica_id = get_start_replica_id(results_path)

if use_pdf_shift and pdf_shift_mode == "sequential":
    required_pdf_replica_ids = range(start_replica_id + 1, start_replica_id + N_replicas + 1)
    missing_pdf_replica_ids = [pdf_id for pdf_id in required_pdf_replica_ids if pdf_id not in pdf_shift_replicas]
    if missing_pdf_replica_ids:
        preview = missing_pdf_replica_ids[:10]
        suffix = "..." if len(missing_pdf_replica_ids) > 10 else ""
        raise ValueError(
            f"Sequential PDF-shift mode requires PDF replicas {start_replica_id + 1} through {start_replica_id + N_replicas}, "
            f"but these IDs are missing under {pdf_diff_root}: {preview}{suffix}"
        )


In [18]:
replica_results = []
replica_ids = range(start_replica_id, start_replica_id + N_replicas)

for replica_id in tqdm(replica_ids, desc="Replica refits"):
    data_list = generate_experimental_replica_data(replica_rng)
    current_pdf_replica_id, current_pdf_shift_list = sample_pdf_replica(replica_rng, replica_id)
    theta0_replica = draw_theta0(replica_rng)

    res = pybobyqa.solve(
        objective_log,
        theta0_replica,
        bounds=(np.zeros(dim), np.ones(dim)),
        maxfun=maxfun,
        rhobeg=rhobeg,
        rhoend=rhoend,
        scaling_within_bounds=True,
        seek_global_minimum=True,
    )

    best_chi2dN = float(10 ** res.f)
    optimal_params_free = denormalize_params(res.x)

    full_params = initial_params.copy()
    full_params[free_idx] = optimal_params_free
    full_params[frozen_idx] = frozen_vals

    row = {
        "replica_id": replica_id,
        "pdf_replica_id": current_pdf_replica_id,
        "success": int(res.flag == 0),
        "nfev": int(res.nf),
        "best_chi2dN": best_chi2dN,
    }
    for name, value in zip(param_columns, full_params):
        row[name] = float(value)

    replica_results.append(row)

    fmt = lambda x: f"{float(x):.8g}"
    with results_path.open("a", newline="") as f:
        csv.writer(f).writerow(
            [
                row["replica_id"],
                row["pdf_replica_id"],
                row["success"],
                row["nfev"],
                fmt(row["best_chi2dN"]),
                *[fmt(row[name]) for name in param_columns],
            ]
        )

    pdf_label = current_pdf_replica_id if current_pdf_replica_id is not None else "off"
    print(
        f"Replica {replica_id}: pdf={pdf_label}, "
        f"chi2/N={best_chi2dN:.3f}, nfev={int(res.nf)}"
    )

data_list = clone_data_list(nominal_data_list)
current_pdf_replica_id = None
current_pdf_shift_list = zero_pdf_shift_list

replica_results_df = pd.DataFrame(replica_results)


Replica refits:   2%|▏         | 1/60 [06:50<6:43:15, 410.09s/it]

Replica 40: pdf=41, chi2/N=1.901, nfev=5000


Replica refits:   3%|▎         | 2/60 [13:40<6:36:35, 410.27s/it]

Replica 41: pdf=42, chi2/N=1.665, nfev=5000


Replica refits:   5%|▌         | 3/60 [20:31<6:30:12, 410.74s/it]

...

Replica 97: pdf=98, chi2/N=1.673, nfev=5000


Replica refits:  98%|█████████▊| 59/60 [7:09:40<07:08, 428.44s/it]

Replica 98: pdf=99, chi2/N=1.758, nfev=5000


Replica refits: 100%|██████████| 60/60 [7:16:49<00:00, 436.83s/it]

Replica 99: pdf=100, chi2/N=1.870, nfev=5000


In [19]:
display(replica_results_df.head())

print(f"Wrote {len(replica_results_df)} replica refits to {results_path}")
print(f"Replica ids: {start_replica_id} to {start_replica_id + len(replica_results_df) - 1}")
print()
print(replica_results_df[["best_chi2dN", "nfev"]].describe())

if use_pdf_shift:
    print()
    print(f"PDF shift mode: {pdf_shift_mode}")
    print("PDF replica usage:")
    display(replica_results_df["pdf_replica_id"].value_counts().sort_index().rename("count").to_frame())
else:
    print()
    print("PDF replica shifts were disabled for this run.")


,replica_id,pdf_replica_id,success,nfev,best_chi2dN,param_0,param_1,param_2,param_3,param_4,param_5,param_6,param_7,param_8,param_9,param_10
0,40,41,0,5000,1.900861,0.514681,1.123,0.867378,3.400976,-0.158479,0.0,0.0,0.402507,-0.078475,0.0,0.0
1,41,42,0,5000,1.665000,0.501794,1.123,0.944819,2.093737,0.140970,0.0,0.0,-3.357648,0.101166,0.0,0.0
2,42,43,0,5000,1.818014,0.494715,1.123,0.914264,-3.230537,0.011077,0.0,0.0,-2.326023,-0.125323,0.0,0.0
3,43,44,0,5000,1.594509,0.510884,1.123,0.927550,3.052390,-0.052790,0.0,0.0,-3.063674,-0.070788,0.0,0.0
4,44,45,0,5000,1.783770,0.483554,1.123,0.976074,1.965093,0.378386,0.0,0.0,-5.287429,0.270121,0.0,0.0


Wrote 60 replica refits to replica_data\replica_0311.csv
Replica ids: 40 to 99

       best_chi2dN    nfev
count    60.000000    60.0
mean      1.752788  5000.0
std       0.143669     0.0
min       1.445406  5000.0
25%       1.660015  5000.0
50%       1.733508  5000.0
75%       1.846127  5000.0
max       2.338665  5000.0

PDF shift mode: sequential
PDF replica usage:


,count
pdf_replica_id,
41,1
42,1
43,1
44,1
45,1
46,1
47,1
48,1
49,1
